# **Question 3.6.1**

**1.This section directly implements the softmax function based on the mathematical definition of the softmax operation. What problems might this cause? Hint: Try calculating the value of exp(50).**<br><br>
If you implement softmax strictly according to its mathematical definition, exp(x) will exceed the range of floating-point representation when the input value is large, causing an overflow (resulting in inf) and rendering the entire softmax calculation invalid (returning nan). The solution is to subtract the maximum value of each row before performing the exponential operation.
$$softmax(x_i) = \frac{exp(x_i - max(x))}{\sum_{j}exp(x_j - max(x))}$$
There are two benefits to doing this:
1. **Numerical stability: After subtracting the maximum value, the maximum input for the exponential function is 0 (since exp(0) = 1), and all other terms are less than or equal to 1, completely avoiding overflow.**
2. **Mathematical equivalence: Since both the numerator and denominator are multiplied by exp(-max(x)), the result is exactly the same as the original definition.**

# **Question 3.6.2**

**1.The `cross_entropy` function in this section is implemented based on the definition of the cross-entropy loss function. What might be wrong with it? Hint: Consider the domain of the logarithm.**<br><br>
The cross-entropy loss function is defined by the log function. When the model predicts a probability of 0, the value of the log function is negative infinity.<br><br>
The key to solving this problem lies in combining the softmax and log operations into a single operation and performing the calculation directly in the log domain—this is log_softmax:
$$log(softmax(x_i)) = x_i - max(x) - log\left(\sum\limits_{j}exp(x_j - max(x))\right)$$
This formula has several advantages:
1. **The calculated probability is not displayed, so there is no situation where the probability underflows to 0 and is then logged.**
2. **Even if a particular $x_i-max(x)$ is very small, it remains in the expression, and the final result is a finite negative number rather than -inf.**

# **Question 3.6.3**
Please propose a solution to address the two issues mentioned above.(For more details, see the previous two questions.)

# **Question 3.6.4**

**1.Is the classification label with the highest probability of return always the optimal solution? For example, can this approach be used in medical diagnosis scenarios?**
* **Ignoring differences in confidence**: argmax focuses only on the category with the highest probability, while ignoring **the absolute probability value of that prediction**. For example, a model predicting a 51% probability of “illness” and a 49% probability of “health” is a completely different scenario from one predicting a 99% probability of “illness.”
* **Cost Asymmetry**: **In medical diagnosis, the cost of a missed diagnosis (false negative) is far higher than that of a misdiagnosis (false positive)**. Simply returning the label with the highest probability does not reflect this difference in cost. A more reasonable approach is to adjust the decision threshold based on cost.
* **In cases of ties or close results**: When the predicted probabilities across multiple classes are very close, the argmax selection may be highly random, and returning a single label is not robust in such situations.
* **Quantifying Uncertainty Is Essential**: In critical decision-making, understanding a model’s uncertainty is crucial. Simply returning a single label does not provide sufficient information. Ideally, the output should include **a complete probability distribution** or **confidence scores** for decision-makers to consider.

# **Question 3.6.5**

**1.Suppose we use softmax regression to predict the next word. What problems might arise if there are too many possible words to choose from?**
* **Computational cost skyrockets**: The softmax calculation requires summing the exponents for all classes, and **the computational effort is proportional to the size of the vocabulary**.
* **Probability distributions are “diluted”**: The softmax function outputs a probability distribution where the sum of the probabilities of all words equals 1. When the number of possible words is extremely large, **the probability assigned to each word becomes minuscule, and the differences in probability between words are negligible**, making it difficult for the model to effectively distinguish which word is truly the next one. This results in the probabilities of **a large number of words being close to 0**, leading to significant computational waste.
* **Data Sparsity and the Risk of Overfitting**: In natural language, word frequencies follow Zipf’s Law, and **the vast majority of words occur very rarely**. In an effort to fit these sparse, rare words, models tend to learn noise from the data, leading to **overfitting and poor generalization**.